# ForestSim Semantic Segmentation – Qualitative Evaluation on Spot Data

This notebook complements the ForestSim dataset exploration notebook with a qualitative, inference-only evaluation of segmentation models on real images and video captured by Boston Dynamics' Spot robot. No training or fine-tuning is performed here: the goal is to check how well models trained on synthetic data (VIS-ForestSim) or on generic datasets (Mask2Former, SegFormer) generalize to real, unstructured forest imagery acquired by Spot's onboard cameras.

Four experiments are carried out:
1. Inference with **VIS-ForestSim** (DeepLabV3, ResNet-50 backbone) on a single real image from Spot.
2. A qualitative comparison between VIS-ForestSim and **Mask2Former** (Swin-Large, ADE20K) on the same image.
3. Frame-by-frame inference with VIS-ForestSim on a **short video clip** from Spot, to assess temporal stability and runtime performance.
4. A further qualitative comparison with **SegFormer-B5** (ADE20K), used here purely as an out-of-the-box baseline with no fine-tuning, just to get a first sense of how much potential a lighter, transformer-based architecture might have.

## VIS-ForestSim Inference on a Real Spot Image

This first experiment loads the VIS-ForestSim checkpoint from Hugging Face (DeepLabV3 with a ResNet-50 backbone, 24 semantic classes) and runs inference on a single RGB frame captured by Spot's right fisheye camera. The image is resized to 512x512 for inference, and the predicted mask is then upsampled with nearest-neighbor interpolation back to the original resolution before being colorized and overlaid on the source image.

**Result.** The predicted mask is dominated by `generic_ground` (60.25%) and `tree` (35.23%), with `rock` (2.00%), `grass` (1.25%) and `sky` (0.52%) as the next most represented classes, plus small traces of `water`, `bush`, `bridge`, `concrete` and `building`. This is consistent with the qualitative behavior observed more broadly on real Spot footage: the model correctly separates trees and sky from the ground, but tends to fold most of the traversable terrain into the generic `generic_ground` class rather than distinguishing `grass` from other ground types, and it occasionally confuses rock-like textures (tree trunk bases, roots, shadowed bark) with the `rock` class.

In [ ]:
# ==========================================
# VIS-FORESTSIM
# ==========================================

!pip -q install huggingface_hub

import torch
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from huggingface_hub import hf_hub_download
from torchvision import transforms
from torchvision.models.segmentation import deeplabv3_resnet50

# --------------------------------------------------
# CONFIG
# --------------------------------------------------

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CLASSES = [
    "grass",
    "tree",
    "pole",
    "water",
    "sky",
    "vehicle",
    "container",
    "asphalt",
    "gravel",
    "mulch",
    "rockbed",
    "log",
    "bicycle",
    "person",
    "fence",
    "bush",
    "sign",
    "rock",
    "bridge",
    "concrete",
    "table",
    "building",
    "void",
    "generic_ground"
]

# RGB palette
PALETTE = np.array([
    [ 34,139, 34],  # grass
    [  0,100,  0],  # tree
    [255,255,  0],  # pole
    [ 30,144,255],  # water
    [135,206,235],  # sky
    [255,  0,  0],  # vehicle
    [255,140,  0],  # container
    [128,128,128],  # asphalt
    [210,180,140],  # gravel
    [160, 82, 45],  # mulch
    [112,128,144],  # rockbed
    [139, 69, 19],  # log
    [255, 20,147],  # bicycle
    [255,105,180],  # person
    [184,134, 11],  # fence
    [ 50,205, 50],  # bush
    [255,215,  0],  # sign
    [105,105,105],  # rock
    [ 72, 61,139],  # bridge
    [169,169,169],  # concrete
    [218,165, 32],  # table
    [128,  0,128],  # building
    [  0,  0,  0],  # void
    [205,133, 63],  # generic ground
], dtype=np.uint8)

# --------------------------------------------------
# DOWNLOAD MODEL
# --------------------------------------------------

print("Downloading checkpoint...")

ckpt_path = hf_hub_download(
    repo_id="ilessio-aiflowlab/project_vis_forestsim",
    filename="pytorch/vis_forestsim_v1.pth"
)

print("Loading model...")

model = deeplabv3_resnet50(
    num_classes=24,
    weights=None,
    weights_backbone=None
)

ckpt = torch.load(ckpt_path, map_location="cpu")

if "model" in ckpt:
    model.load_state_dict(ckpt["model"])
else:
    model.load_state_dict(ckpt)

model = model.to(DEVICE)
model.eval()

# --------------------------------------------------
# PREPROCESS
# --------------------------------------------------

from google.colab import files
uploaded = files.upload()
IMAGE_PATH = list(uploaded.keys())[0]

img = Image.open(IMAGE_PATH).convert("RGB")
orig_w, orig_h = img.size

transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),

])

x = transform(img).unsqueeze(0).to(DEVICE)

# --------------------------------------------------
# INFERENCE
# --------------------------------------------------

with torch.no_grad():
    logits = model(x)["out"]
    pred = logits.argmax(1)[0].cpu().numpy()

# --------------------------------------------------
# UPSAMPLE TO ORIGINAL SIZE
# --------------------------------------------------

mask_img = Image.fromarray(pred.astype(np.uint8))
mask_img = mask_img.resize((orig_w, orig_h), Image.NEAREST)

mask = np.array(mask_img)

# --------------------------------------------------
# COLORIZE
# --------------------------------------------------

color_mask = PALETTE[mask]

img_np = np.array(img)

overlay = (
    0.55 * img_np +
    0.45 * color_mask
).astype(np.uint8)

# --------------------------------------------------
# CLASS STATISTICS
# --------------------------------------------------

print("\nDetected classes:\n")

unique, counts = np.unique(mask, return_counts=True)

for cls, cnt in zip(unique, counts):
    perc = 100 * cnt / mask.size
    print(f"{cls:2d} | {CLASSES[cls]:15s} | {perc:6.2f}%")

# --------------------------------------------------
# VISUALIZATION + LEGEND
# --------------------------------------------------

from matplotlib.patches import Patch

present_classes = sorted(unique)

legend_elements = [
    Patch(
        facecolor=PALETTE[c] / 255.0,
        edgecolor='black',
        label=f"{CLASSES[c]} ({100*counts[list(unique).index(c)]/mask.size:.1f}%)"
    )
    for c in present_classes
]

fig = plt.figure(figsize=(20, 12))

gs = fig.add_gridspec(
    2, 3,
    height_ratios=[4, 1]
)

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[0, 2])

ax1.imshow(img_np)
ax1.set_title("Original Image")
ax1.axis("off")

ax2.imshow(color_mask)
ax2.set_title("Semantic Segmentation")
ax2.axis("off")

ax3.imshow(overlay)
ax3.set_title("Overlay")
ax3.axis("off")

legend_ax = fig.add_subplot(gs[1, :])
legend_ax.axis("off")

legend_ax.legend(
    handles=legend_elements,
    loc="center",
    ncol=4,
    fontsize=10,
    frameon=True
)

plt.tight_layout()
plt.show()




## Comparison with Mask2Former (Swin-Large, ADE20K)

To put VIS-ForestSim's behavior in context, this section runs the general-purpose Mask2Former model (Swin-Large backbone, trained on ADE20K) on the same image and displays both predictions side by side, each with its own class legend.

The two models show complementary strengths: VIS-ForestSim, being trained specifically on forest-like synthetic scenes, produces a more coherent and detailed segmentation of the tree canopy and vegetation structure, while Mask2Former — trained on a generic, object-centric dataset — is better at recognizing man-made elements (e.g. vehicles, fences, benches) that are not explicitly represented as separate classes in ForestSim. Mask2Former also tends to label grassy ground as `grass` more readily, whereas VIS-ForestSim leans toward the catch-all `generic_ground` class.

In [ ]:
# ==========================================
# COMPARISON: FORESTSIM VS MASK2FORMER (WITH LEGENDS)
# ==========================================

print("Installing missing dependencies for Mask2Former...")
!pip -q install transformers accelerate

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from matplotlib.patches import Patch
from transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Load Mask2Former
print("Loading Mask2Former (Swin-Large)...")
processor = AutoImageProcessor.from_pretrained("facebook/mask2former-swin-large-ade-semantic")
m2f_model = Mask2FormerForUniversalSegmentation.from_pretrained("facebook/mask2former-swin-large-ade-semantic")
m2f_model = m2f_model.to(DEVICE)
m2f_model.eval()

try:
    _ = img.size
except NameError:
    img = Image.open(IMAGE_PATH).convert("RGB")

orig_w, orig_h = img.size

print("Running inference on Mask2Former...")
inputs = processor(images=img, return_tensors="pt").to(DEVICE)

with torch.no_grad():
    outputs = m2f_model(**inputs)

predicted_semantic_map = processor.post_process_semantic_segmentation(
    outputs,
    target_sizes=[(orig_h, orig_w)]
)[0]

m2f_mask = predicted_semantic_map.cpu().numpy()

#
labels_info = m2f_model.config.id2label
num_classes_m2f = len(labels_info)

# Fixed palette for consistent coloring
np.random.seed(42)
m2f_palette = np.random.randint(0, 255, size=(num_classes_m2f, 3), dtype=np.uint8)
if 0 in labels_info: m2f_palette[0] = [0, 0, 0]

m2f_color_mask = m2f_palette[m2f_mask]

img_np = np.array(img)
m2f_overlay = (0.55 * img_np + 0.45 * m2f_color_mask).astype(np.uint8)

# Compute detected Mask2Former classes for the legend
m2f_unique, m2f_counts = np.unique(m2f_mask, return_counts=True)
m2f_present_classes = sorted(m2f_unique)

m2f_legend_elements = [
    Patch(
        facecolor=m2f_palette[c] / 255.0,
        edgecolor='black',
        label=f"{labels_info[c]} ({100*m2f_counts[list(m2f_unique).index(c)]/m2f_mask.size:.1f}%)"
    )
    for c in m2f_present_classes
]

# Retrieve ForestSim data (from the previous cell)
try:
    forestsim_overlay = overlay
    forestsim_mask_color = color_mask
    forestsim_legend_elements = legend_elements
except NameError:
    print("[WARNING] ForestSim data not found. Showing only Mask2Former.")
    forestsim_overlay = None

# ==========================================
# COMPARATIVE VISUALIZATION WITH LEGENDS
# ==========================================

print("Generating comparison plot...")

if forestsim_overlay is not None:
    # Grid layout: 2 rows of images + 2 thin rows for the respective legends
    fig = plt.figure(figsize=(24, 18))
    gs = fig.add_gridspec(4, 3, height_ratios=[4, 0.8, 4, 0.8])

    # --- ROW 1: ForestSim Images ---
    ax_f1 = fig.add_subplot(gs[0, 0])
    ax_f1.imshow(img_np)
    ax_f1.set_title("Original Image", fontsize=14, fontweight='bold')
    ax_f1.axis("off")

    ax_f2 = fig.add_subplot(gs[0, 1])
    ax_f2.imshow(forestsim_mask_color)
    ax_f2.set_title("ForestSim - Semantic Map", fontsize=14, fontweight='bold')
    ax_f2.axis("off")

    ax_f3 = fig.add_subplot(gs[0, 2])
    ax_f3.imshow(forestsim_overlay)
    ax_f3.set_title("ForestSim - Overlay", fontsize=14, fontweight='bold')
    ax_f3.axis("off")

    # --- ROW 2: ForestSim Legend ---
    legend_f_ax = fig.add_subplot(gs[1, :])
    legend_f_ax.axis("off")
    legend_f_ax.legend(
        handles=forestsim_legend_elements,
        loc="center", ncol=5, fontsize=11, frameon=True, title="ForestSim Classes"
    )

    # --- ROW 3: Mask2Former Images ---
    ax_m1 = fig.add_subplot(gs[2, 0])
    ax_m1.imshow(img_np)
    ax_m1.set_title("Original Image", fontsize=14, fontweight='bold')
    ax_m1.axis("off")

    ax_m2 = fig.add_subplot(gs[2, 1])
    ax_m2.imshow(m2f_color_mask)
    ax_m2.set_title("Mask2Former - Semantic Map", fontsize=14, fontweight='bold')
    ax_m2.axis("off")

    ax_m3 = fig.add_subplot(gs[2, 2])
    ax_m3.imshow(m2f_overlay)
    ax_m3.set_title("Mask2Former - Overlay (Swin-Large)", fontsize=14, fontweight='bold')
    ax_m3.axis("off")

    # --- ROW 4: Mask2Former Legend ---
    legend_m_ax = fig.add_subplot(gs[3, :])
    legend_m_ax.axis("off")
    legend_m_ax.legend(
        handles=m2f_legend_elements,
        loc="center", ncol=5, fontsize=11, frameon=True, title="Mask2Former (ADE20K) Classes"
    )

else:
    # Fallback if ForestSim data is not present
    fig = plt.figure(figsize=(24, 10))
    gs = fig.add_gridspec(2, 3, height_ratios=[4, 1])

    ax1 = fig.add_subplot(gs[0, 0])
    ax1.imshow(img_np)
    ax1.axis("off")

    ax2 = fig.add_subplot(gs[0, 1])
    ax2.imshow(m2f_color_mask)
    ax2.set_title("Mask2Former - Semantic Map", fontsize=14)
    ax2.axis("off")

    ax3 = fig.add_subplot(gs[0, 2])
    ax3.imshow(m2f_overlay)
    ax3.set_title("Mask2Former - Overlay", fontsize=14)
    ax3.axis("off")

    legend_ax = fig.add_subplot(gs[1, :])
    legend_ax.axis("off")
    legend_ax.legend(handles=m2f_legend_elements, loc="center", ncol=5, fontsize=11, frameon=True)

plt.tight_layout()
plt.show()

## Video Inference: Temporal Consistency and Runtime Performance

This experiment applies VIS-ForestSim frame-by-frame to a short video clip recorded by Spot, to evaluate two aspects that a single-image test cannot capture: the temporal stability of the predicted masks across consecutive frames, and the runtime performance of the model on a real GPU.

Each frame is processed independently (no temporal smoothing), and the resulting segmentation and overlay videos are exported, together with down-sampled GIF previews for quick visual inspection directly in the notebook.

**Result.** On an NVIDIA T4 GPU, the 640x480, 80-frame clip (recorded at ~10.04 FPS) is processed in 11.24 s, corresponding to an average inference time of 140.44 ms/frame, i.e. ~7.12 FPS — below the source video's frame rate, so the model does not currently run in real time on this hardware. Qualitatively, the predictions are fairly stable over time: the ground is consistently classified as `generic_ground`, and most flickering is limited to small grass tufts near the camera, which are intermittently classified as `grass` due to lighting changes between frames. A more interesting effect appears on partially shaded background vegetation, where the illuminated part of a hedge is stably classified as `tree` while the shaded part is frequently mislabeled as `generic_ground` — suggesting the model is sensitive to illumination and low-contrast textures. This runtime result is an important data point for the thesis: even the teacher network alone is not real-time on embedded-class hardware, which reinforces the motivation for training a lighter, distilled student model for on-robot deployment.

In [ ]:
# ==========================================
# VIS-FORESTSIM VIDEO SEGMENTATION + GIF PREVIEW
# ==========================================

!pip -q install huggingface_hub opencv-python imageio

import cv2
import time
import torch
import imageio.v2 as imageio
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from google.colab import files
from huggingface_hub import hf_hub_download
from torchvision import transforms
from torchvision.models.segmentation import deeplabv3_resnet50

from IPython.display import HTML, display
from base64 import b64encode

# --------------------------------------------------
# CONFIG
# --------------------------------------------------

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MAX_GIF_FRAMES = 150

CLASSES = [
    "grass","tree","pole","water","sky","vehicle",
    "container","asphalt","gravel","mulch","rockbed",
    "log","bicycle","person","fence","bush","sign",
    "rock","bridge","concrete","table","building",
    "void","generic_ground"
]

PALETTE = np.array([
    [ 34,139, 34],  # grass
    [  0,100,  0],  # tree
    [255,255,  0],  # pole
    [ 30,144,255],  # water
    [135,206,235],  # sky
    [255,  0,  0],  # vehicle
    [255,140,  0],  # container
    [128,128,128],  # asphalt
    [210,180,140],  # gravel
    [160, 82, 45],  # mulch
    [112,128,144],  # rockbed
    [139, 69, 19],  # log
    [255, 20,147],  # bicycle
    [255,105,180],  # person
    [184,134, 11],  # fence
    [ 50,205, 50],  # bush
    [255,215,  0],  # sign
    [105,105,105],  # rock
    [ 72, 61,139],  # bridge
    [169,169,169],  # concrete
    [218,165, 32],  # table
    [128,  0,128],  # building
    [  0,  0,  0],  # void
    [205,133, 63],  # generic ground
], dtype=np.uint8)

# --------------------------------------------------
# LOAD MODEL
# --------------------------------------------------

print("Downloading checkpoint...")

ckpt_path = hf_hub_download(
    repo_id="ilessio-aiflowlab/project_vis_forestsim",
    filename="pytorch/vis_forestsim_v1.pth"
)

print("Loading model...")

model = deeplabv3_resnet50(
    num_classes=24,
    weights=None,
    weights_backbone=None
)

ckpt = torch.load(ckpt_path, map_location="cpu")

if "model" in ckpt:
    model.load_state_dict(ckpt["model"])
else:
    model.load_state_dict(ckpt)

model = model.to(DEVICE)
model.eval()

print(f"Model loaded on {DEVICE}")

# --------------------------------------------------
# VIDEO UPLOAD
# --------------------------------------------------

uploaded = files.upload()
VIDEO_PATH = list(uploaded.keys())[0]

cap = cv2.VideoCapture(VIDEO_PATH)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Resolution : {width}x{height}")
print(f"FPS        : {fps:.2f}")
print(f"Frames     : {total_frames}")

# --------------------------------------------------
# OUTPUT VIDEOS
# --------------------------------------------------

overlay_writer = cv2.VideoWriter(
    "forestsim_overlay.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

mask_writer = cv2.VideoWriter(
    "forestsim_mask.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

# --------------------------------------------------
# PREPROCESS
# --------------------------------------------------

transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
])

# --------------------------------------------------
# GIF SAMPLING
# --------------------------------------------------

sample_step = max(1, total_frames // MAX_GIF_FRAMES)

orig_gif = []
mask_gif = []
overlay_gif = []

# --------------------------------------------------
# INFERENCE
# --------------------------------------------------

start_time = time.time()

frame_idx = 0

while True:

    ret, frame = cap.read()

    if not ret:
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    pil_img = Image.fromarray(rgb)

    x = transform(pil_img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits = model(x)["out"]
        pred = logits.argmax(1)[0].cpu().numpy()

    mask = Image.fromarray(pred.astype(np.uint8))
    mask = mask.resize((width, height), Image.NEAREST)
    mask = np.array(mask)

    color_mask = PALETTE[mask]

    overlay = (
        0.55 * rgb +
        0.45 * color_mask
    ).astype(np.uint8)

    overlay_writer.write(
        cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR)
    )

    mask_writer.write(
        cv2.cvtColor(color_mask, cv2.COLOR_RGB2BGR)
    )

    if frame_idx % sample_step == 0:

        orig_gif.append(rgb)

        mask_gif.append(color_mask)

        overlay_gif.append(overlay)

    frame_idx += 1

    if frame_idx % 50 == 0:
        print(f"{frame_idx}/{total_frames}")

cap.release()
overlay_writer.release()
mask_writer.release()

# --------------------------------------------------
# PERFORMANCE
# --------------------------------------------------

total_time = time.time() - start_time

effective_fps = frame_idx / total_time
ms_per_frame = 1000.0 / effective_fps

# --------------------------------------------------
# CREATE GIFS
# --------------------------------------------------

print("\nCreating GIF previews...")

gif_fps = min(10, fps)

imageio.mimsave(
    "original.gif",
    orig_gif,
    fps=gif_fps
)

imageio.mimsave(
    "mask.gif",
    mask_gif,
    fps=gif_fps
)

imageio.mimsave(
    "overlay.gif",
    overlay_gif,
    fps=gif_fps
)

# --------------------------------------------------
# PERFORMANCE REPORT
# --------------------------------------------------

print("\n==============================")
print("PERFORMANCE REPORT")
print("==============================")

print(f"Device           : {DEVICE}")
print(f"Resolution       : {width} x {height}")
print(f"Frames           : {frame_idx}")
print(f"Video FPS        : {fps:.2f}")

print(f"\nTotal Time       : {total_time:.2f} s")
print(f"Inference FPS    : {effective_fps:.2f}")
print(f"Time / Frame     : {ms_per_frame:.2f} ms")

if effective_fps >= fps:
    print("\nReal-time capable ✓")
else:
    print("\nNot real-time ✗")

# --------------------------------------------------
# DISPLAY GIFS
# --------------------------------------------------

def show_gif(path, width=450):

    data_url = b64encode(
        open(path, "rb").read()
    ).decode()

    return HTML(
        f'<img src="data:image/gif;base64,{data_url}" width="{width}">'
    )

print("\nORIGINAL VIDEO")
display(show_gif("original.gif"))

print("\nSEGMENTATION")
display(show_gif("mask.gif"))

print("\nOVERLAY")
display(show_gif("overlay.gif"))



### Exporting Frames for Further Analysis

The GIF previews generated above are split back into individual PNG frames and packed into zip archives (original / mask / overlay), to make it easier to inspect specific frames outside Colab or to reuse them for other analyses.

In [ ]:
from PIL import Image
import os
import shutil
from google.colab import files

def gif_to_frames(gif_path, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    gif = Image.open(gif_path)

    frame = 0
    while True:
        try:
            gif.seek(frame)

            gif.convert("RGB").save(
                os.path.join(output_dir, f"frame_{frame:03d}.png")
            )

            frame += 1

        except EOFError:
            break

    print(f"Extracted {frame} frames in {output_dir}")

gif_to_frames("original.gif", "frames_original")
gif_to_frames("mask.gif", "frames_mask")
gif_to_frames("overlay.gif", "frames_overlay")

folders_to_download = ["frames_original", "frames_mask", "frames_overlay"]

for folder in folders_to_download:
    archive_name = shutil.make_archive(folder, 'zip', folder)
    files.download(archive_name)


## Comparison with SegFormer-B5 (Out-of-the-Box Baseline)

As a further point of comparison, this section runs SegFormer-B5 (`nvidia/segformer-b5-finetuned-ade-640-640`) on the same image, again with its own random color palette and class legend. This is only a quick, **out-of-the-box trial**: no fine-tuning was performed on ForestSim or on Spot data. The goal is simply to get a first qualitative sense of how a transformer-based, ADE20K-pretrained model behaves on this kind of scene, as a possible alternative backbone to consider — alongside MobileNetV3, EfficientNet or SegFormer-B0 — when distilling a lighter student model from VIS-ForestSim.

Since SegFormer was not adapted to the ForestSim class taxonomy, a direct numeric comparison with VIS-ForestSim's classes is not meaningful here; the comparison should be read qualitatively, focusing on how well object boundaries and vegetation structure are captured, similarly to the Mask2Former comparison above.

In [ ]:
# ==========================================
# COMPARISON: FORESTSIM VS SEGFORMER-B5
# ==========================================

print("Installing missing dependencies...")
!pip -q install transformers accelerate

import torch
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from matplotlib.patches import Patch
from transformers import (
    SegformerImageProcessor,
    SegformerForSemanticSegmentation
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --------------------------------------------------
# LOAD SEGFORMER
# --------------------------------------------------

MODEL_NAME = "nvidia/segformer-b5-finetuned-ade-640-640"

print("Loading SegFormer B5...")

processor = SegformerImageProcessor.from_pretrained(
    MODEL_NAME
)

segformer_model = SegformerForSemanticSegmentation.from_pretrained(
    MODEL_NAME
)

segformer_model = segformer_model.to(DEVICE)
segformer_model.eval()

# --------------------------------------------------
# IMAGE
# --------------------------------------------------

try:
    _ = img.size
except NameError:
    img = Image.open(IMAGE_PATH).convert("RGB")
    #img = img.rotate(-90, expand=True)

orig_w, orig_h = img.size

# --------------------------------------------------
# INFERENCE
# --------------------------------------------------

print("Running SegFormer inference...")

inputs = processor(
    images=img,
    return_tensors="pt"
).to(DEVICE)

with torch.no_grad():

    outputs = segformer_model(**inputs)

logits = outputs.logits

logits = torch.nn.functional.interpolate(
    logits,
    size=(orig_h, orig_w),
    mode="bilinear",
    align_corners=False
)

segformer_mask = logits.argmax(dim=1)[0].cpu().numpy()

# --------------------------------------------------
# COLOR PALETTE
# --------------------------------------------------

labels_info = segformer_model.config.id2label

num_classes = len(labels_info)

np.random.seed(42)

segformer_palette = np.random.randint(
    0,
    255,
    size=(num_classes, 3),
    dtype=np.uint8
)

if 0 in labels_info:
    segformer_palette[0] = [0,0,0]

segformer_color_mask = segformer_palette[
    segformer_mask
]

img_np = np.array(img)

segformer_overlay = (
    0.55 * img_np +
    0.45 * segformer_color_mask
).astype(np.uint8)

# --------------------------------------------------
# SEGFORMER LEGEND
# --------------------------------------------------

unique_seg, counts_seg = np.unique(
    segformer_mask,
    return_counts=True
)

present_seg = []

for c, cnt in zip(unique_seg, counts_seg):

    perc = 100 * cnt / segformer_mask.size

    if perc > 0.3:
        present_seg.append((c, perc))

segformer_legend_elements = []

for c, perc in present_seg:

    segformer_legend_elements.append(

        Patch(
            facecolor=segformer_palette[c] / 255.0,
            edgecolor='black',
            label=f"{labels_info[c]} ({perc:.1f}%)"
        )

    )

# --------------------------------------------------
# RECOVER FORESTSIM RESULTS
# --------------------------------------------------

try:

    forestsim_overlay = overlay
    forestsim_mask_color = color_mask
    forestsim_legend_elements = legend_elements

except NameError:

    print(
        "[WARNING] ForestSim results not found."
    )

    forestsim_overlay = None

# --------------------------------------------------
# VISUALIZATION
# --------------------------------------------------

print("Generating comparison plot...")

if forestsim_overlay is not None:

    fig = plt.figure(
        figsize=(24,18)
    )

    gs = fig.add_gridspec(
        4,
        3,
        height_ratios=[4,0.8,4,0.8]
    )

    # =====================================
    # FORESTSIM
    # =====================================

    ax_f1 = fig.add_subplot(gs[0,0])
    ax_f1.imshow(img_np)
    ax_f1.set_title(
        "Original Image",
        fontsize=14,
        fontweight="bold"
    )
    ax_f1.axis("off")

    ax_f2 = fig.add_subplot(gs[0,1])
    ax_f2.imshow(forestsim_mask_color)
    ax_f2.set_title(
        "ForestSim - Semantic Map",
        fontsize=14,
        fontweight="bold"
    )
    ax_f2.axis("off")

    ax_f3 = fig.add_subplot(gs[0,2])
    ax_f3.imshow(forestsim_overlay)
    ax_f3.set_title(
        "ForestSim - Overlay",
        fontsize=14,
        fontweight="bold"
    )
    ax_f3.axis("off")

    legend_f_ax = fig.add_subplot(gs[1,:])

    legend_f_ax.axis("off")

    legend_f_ax.legend(
        handles=forestsim_legend_elements,
        loc="center",
        ncol=5,
        fontsize=11,
        frameon=True,
        title="ForestSim Classes"
    )

    # =====================================
    # SEGFORMER
    # =====================================

    ax_s1 = fig.add_subplot(gs[2,0])
    ax_s1.imshow(img_np)
    ax_s1.set_title(
        "Original Image",
        fontsize=14,
        fontweight="bold"
    )
    ax_s1.axis("off")

    ax_s2 = fig.add_subplot(gs[2,1])
    ax_s2.imshow(segformer_color_mask)
    ax_s2.set_title(
        "SegFormer-B5 Semantic Map",
        fontsize=14,
        fontweight="bold"
    )
    ax_s2.axis("off")

    ax_s3 = fig.add_subplot(gs[2,2])
    ax_s3.imshow(segformer_overlay)
    ax_s3.set_title(
        "SegFormer-B5 Overlay",
        fontsize=14,
        fontweight="bold"
    )
    ax_s3.axis("off")

    legend_s_ax = fig.add_subplot(gs[3,:])

    legend_s_ax.axis("off")

    legend_s_ax.legend(
        handles=segformer_legend_elements,
        loc="center",
        ncol=5,
        fontsize=11,
        frameon=True,
        title="SegFormer (ADE20K) Classes"
    )

else:

    fig = plt.figure(
        figsize=(24,10)
    )

    gs = fig.add_gridspec(
        2,
        3,
        height_ratios=[4,1]
    )

    ax1 = fig.add_subplot(gs[0,0])
    ax1.imshow(img_np)
    ax1.axis("off")

    ax2 = fig.add_subplot(gs[0,1])
    ax2.imshow(segformer_color_mask)
    ax2.set_title(
        "SegFormer-B5 Semantic Map"
    )
    ax2.axis("off")

    ax3 = fig.add_subplot(gs[0,2])
    ax3.imshow(segformer_overlay)
    ax3.set_title(
        "SegFormer-B5 Overlay"
    )
    ax3.axis("off")

    legend_ax = fig.add_subplot(gs[1,:])

    legend_ax.axis("off")

    legend_ax.legend(
        handles=segformer_legend_elements,
        loc="center",
        ncol=5,
        fontsize=11,
        frameon=True
    )

plt.tight_layout()
plt.show()